# Steam Reviews Data Collection

This notebook handles the collection of Steam game reviews via the Steam API.

**Author:** Michael Theophanopoulos  
**Purpose:** Data collection for interpretable predictions analysis  
**Last Updated:** 2025-10-28

In [1]:
import requests
import pandas as pd
import time
from pathlib import Path
from typing import List, Dict, Optional
import logging

In [2]:
def scrape_steam_reviews(
    app_id: int = 1245620,
    max_reviews: int = 50000,
    language: str = 'english',
    reviews_per_page: int = 100
) -> List[Dict]:
    """
    Scrape game reviews from Steam API.
    
    Parameters
    ----------
    app_id : int
        Steam application ID (default: 1245620 for Elden Ring)
    max_reviews : int
        Maximum number of reviews to collect
    language : str
        Review language filter
    reviews_per_page : int
        Number of reviews per API request (max 100)
        
    Returns
    -------
    List[Dict]
        List of review dictionaries
    """
    all_reviews = []
    cursor = '*'
    start_time = time.time()
    
    logging.info(f"Starting fetching reviews for app_id={app_id}...")
    
    while len(all_reviews) < max_reviews:
        url = f'https://store.steampowered.com/appreviews/{app_id}'
        params = {
            'json': 1,
            'language': language,
            'cursor': cursor,
            'num_per_page': reviews_per_page,
            'filter': 'recent'
        }
        
        try:
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
            
            if not data.get('reviews'):
                logging.warning("No more reviews available")
                break
            
            all_reviews.extend(data['reviews'])
            cursor = data.get('cursor')
            
            if not cursor:
                logging.warning("No cursor returned, ending pagination")
                break
            
            # Rate limiting: 0.3s between requests
            time.sleep(0.3)
            
        except requests.exceptions.RequestException as e:
            logging.error(f"Request failed: {e}")
            time.sleep(5)
    
    final_reviews = all_reviews[:max_reviews]
    elapsed_time = time.time() - start_time
    
    logging.info(f"Fetched {len(final_reviews)} reviews in {elapsed_time:.2f} seconds")
    
    return final_reviews

In [3]:
def process_reviews(reviews: List[Dict]) -> pd.DataFrame:
    """
    Convert raw review data to structured DataFrame.
    
    Parameters
    ----------
    reviews : List[Dict]
        Raw review data from Steam API
        
    Returns
    -------
    pd.DataFrame
        Structured dataset with selected features
    """
    data = []
    
    for review in reviews:
        record = {
            'review_text': review.get('review', ''),
            'recommended': review.get('voted_up', False),
            'playtime_hours': review.get('author', {}).get('playtime_forever', 0) / 60,
            'games_owned': review.get('author', {}).get('num_games_owned', 0),
            'helpful_votes': review.get('votes_helpful', 0),
            'funny_votes': review.get('votes_funny', 0),
        }
        data.append(record)
    
    df = pd.DataFrame(data)
    logging.info(f"Processed {len(df)} reviews into DataFrame")
    
    return df

## Data Collection

In [4]:
# Configuration
APP_ID = 1245620  # Elden Ring
MAX_REVIEWS = 50000
OUTPUT_FILE = 'steam_elden_ring_reviews.csv'

# Collect reviews
reviews = scrape_steam_reviews(app_id=APP_ID, max_reviews=MAX_REVIEWS)

In [5]:
# Process and save
df = process_reviews(reviews)
df.to_csv(OUTPUT_FILE, index=False)
logging.info(f"Dataset saved to {OUTPUT_FILE}")

In [6]:
# Data summary
print(f"Dataset shape: {df.shape}")
print(f"Positive reviews: {df['recommended'].sum()} ({df['recommended'].mean()*100:.1f}%)")
print(f"\nFirst few rows:")
df.head()

Dataset shape: (50000, 6)
Positive reviews: 46993 (94.0%)

First few rows:


,review_text,recommended,playtime_hours,games_owned,helpful_votes,funny_votes
0,this game made me wanna regret my life choices...,True,104.300000,0,0,0
1,10/10,True,35.266667,42,0,0
2,beautiful,True,25.816667,23,0,0
3,"If you liked the Dark Souls trilogy, I'd say t...",True,176.616667,0,0,0
4,"It's rage inducing, yet I cant stop playing. I...",True,79.516667,328,0,0
